# 계획 단계 LoRA 어댑터 학습 — Qwen3-0.6B / 1.7B

시나리오 A 계획 단계(`plan_brief`)의 3키 추출(summary, equipment, materials)을 어댑터로 학습하고, 학습 전(기본 모델)과 바로 비교한다.
어댑터만 저장한다 (기본 모델은 합치지 않는다). 어댑터는 **학습한 기본 모델·같은 판에만** 붙는다.

준비물 (왼쪽 폴더 아이콘으로 올려도 된다):
- `app.zip` — **지금 PC 저장소의 `app/` 폴더를 새로 묶은 것** (**T30 규칙 v2(`apply_rules_v2.py`)를 PC에 적용한 뒤**. 학습 데이터를 서버 검증 코드로 만들기 때문에 서버와 같은 판이어야 한다. 아래 2번 칸이 규칙 v2 가 들어 있는지 확인한다)
- `plan_eval.zip` — 전에 Colab 측정에 올렸던 것 (`scripts/plan_eval/`, `tests/`)
- `plan_train_v2.zip` — 이번 묶음. 이전 `plan_train.zip`·`eval_hard.zip` 내용을 모두 포함한다 (그 둘은 더 올리지 않는다)

런타임 > 런타임 유형 변경에서 GPU(T4면 충분)를 고른다. 처음 한 번은 아래 "스모크"(3분 안팎)로 GPU 학습 경로가 도는지만 본다.

In [ ]:
# 1) 파일 올리기
import os
from google.colab import files

need = ['app.zip', 'plan_eval.zip', 'plan_train_v2.zip']
if not all(os.path.exists('/content/' + n) for n in need):
    files.upload()
print([n for n in need if not os.path.exists('/content/' + n)] or '세 파일 모두 있음')

In [ ]:
# 2) 풀기 → 평가 도구 확장 적용(정확 일치 검사 먼저) → 의존 패키지
!rm -rf /content/repo && mkdir -p /content/repo
!cd /content/repo && unzip -q -o /content/app.zip && unzip -q -o /content/plan_eval.zip && unzip -q -o /content/plan_train_v2.zip
!pip -q install pydantic pydantic-settings pytest peft
# Colab 에 미리 깔린 torchao 가 옛 판(0.10.0)이면 최신 peft 가 LoRA 를 붙일 때 ImportError 로 멈춘다 (T29). 이 작업은 torchao 를 쓰지 않으니 지운다.
!pip -q uninstall -y torchao
%cd /content/repo
!test -f app/agent/plan_adapter.py && echo "plan_adapter.py 있음 (T26 적용된 app.zip)" || echo "!! app.zip 이 옛 판입니다 (plan_adapter.py 없음). PC 의 현재 app/ 을 다시 묶어 올리세요"
# T30 규칙 v2 확인: 아래 세 줄이 모두 '1' 이상이어야 한다 (0 이면 PC 에 apply_rules_v2.py 를 적용한 뒤 app.zip 을 다시 묶어 올린다)
!grep -c "EXTENDED_VOCAB = True" app/agent/text_rules.py
!grep -c "PROHIBIT_NEGATION = True" app/agent/text_rules.py
!grep -c "SUMMARY_WORD_GUARD = True" app/agent/plan_brief.py
!python apply_plan_train_eval.py --root . --check

In [ ]:
# 3) 적용 → 시험 (모델 없이도 도는 시험 + 아주 작은 모델 학습 시험. 전부 통과해야 한다)
!python apply_plan_train_eval.py --root .
!python -B -m pytest -p no:cacheprovider tests/test_plan_eval_harness.py tests/test_plan_eval_adapter.py tests/test_make_plan_train.py tests/test_plan_train_lora.py tests/test_rules_v2.py tests/test_polite_negation.py tests/test_eval_set_user_hard.py -q

In [ ]:
# 4) 학습 데이터 만들기 (같은 seed면 같은 파일). 평가 문장(40 + 여러 줄 12 + 현장·어려운 문장 27)과 겹치는 것은 자동으로 버린다
!python -m scripts.make_plan_train --out data/plan-train
!python -m scripts.make_plan_train --sample 4

In [ ]:
# 5) GPU 확인. 학습 dtype auto = bfloat16 가능하면 bfloat16, 아니면(T4) float16 (손실 스케일러 사용)
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 유형을 GPU로 바꾸세요."
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
print(torch.cuda.get_device_name(0), "→", DTYPE)

## 스모크 (학습 64건·1에폭·생성 5건)

GPU 경로(float16 autocast + 손실 스케일러)는 PC·샌드박스에서 돌려 보지 못했다. 본 학습 전에 여기서 **오류 없이 끝나고 손실이 숫자로 나오는지**만 본다.

In [ ]:
!python -m scripts.plan_train.train_lora --model Qwen/Qwen3-0.6B --data data/plan-train --out adapters/smoke --limit-train 64 --limit-valid 16 --epochs 1 --gen-check 5

## 기준선 (학습 전 기본 모델) — 평가 79문장 (40 + 여러 줄 12 + 현장·어려운 27)

사전·부정 규칙이 바뀌었으므로(T30) 이전 기준선과 숫자를 비교할 수 없다. **같은 규칙으로 다시 잰다.**
- `long`: 지금 서버가 어댑터 없이 보내는 긴 프롬프트
- `short`: 어댑터 학습 때의 짧은 프롬프트를 **어댑터 없이** 기본 모델에 (어댑터 효과와 프롬프트 효과를 가르려는 비교)
결과는 평가 문장 묶음마다 폴더가 다르다 (`out/plan_eval`, `out/plan_eval_multi`, `out/plan_eval_hard`). 시간이 모자라면 1.7B 기준선은 건너뛰어도 된다 (0.6B 어댑터를 평가하는 데는 0.6B 기준선만 필요하다).

In [ ]:
SETS = [
    ('out/plan_eval', None),
    ('out/plan_eval_multi', 'scripts/plan_eval/eval_set_multiline.jsonl'),
    ('out/plan_eval_hard', 'scripts/plan_eval/eval_set_user_hard.jsonl'),
]
MODELS = ['Qwen/Qwen3-0.6B', 'Qwen/Qwen3-1.7B']   # 1.7B 기준선을 건너뛰려면 첫 항목만 남긴다
for model in MODELS:
    for prompt in ['long', 'short']:
        for out, eval_set in SETS:
            extra = f'--eval-set {eval_set}' if eval_set else ''
            !python -m scripts.plan_eval.run_quality --model {model} --dtype {DTYPE} --prompt {prompt} --out {out} {extra}

## 본 학습 — 0.6B

기본값: LoRA r=16 α=32, 학습률 2e-4, 3에폭, 배치 8, 어텐션·MLP 전부. 끝나면 valid 앞 60건을 서버와 같은 방식으로 생성해 JSON·항목 일치 비율을 찍는다
(간이 확인이다. 판단은 아래 평가 40문장·여러 줄 12문장으로 한다).

In [ ]:
!python -m scripts.plan_train.train_lora --model Qwen/Qwen3-0.6B --data data/plan-train --out adapters/plan_brief_0.6b

## 본 학습 — 1.7B (선택: 0.6B 만 서버에 쓸 거면 건너뛴다)

In [ ]:
!python -m scripts.plan_train.train_lora --model Qwen/Qwen3-1.7B --data data/plan-train --out adapters/plan_brief_1.7b

## 어댑터 측정 (서버와 같은 방식으로 붙인다 · 짧은 프롬프트)

In [ ]:
ADAPTERS = [('Qwen/Qwen3-0.6B', 'adapters/plan_brief_0.6b'), ('Qwen/Qwen3-1.7B', 'adapters/plan_brief_1.7b')]
import os
for model, adapter in ADAPTERS:
    if not os.path.isdir(adapter):
        print('건너뜀 (학습 안 함):', adapter)
        continue
    for out, eval_set in SETS:
        extra = f'--eval-set {eval_set}' if eval_set else ''
        !python -m scripts.plan_eval.run_quality --model {model} --dtype {DTYPE} --adapter {adapter} --out {out} {extra}

In [ ]:
# 비교표 (따옴표로 감싼 *는 compare 가 직접 푼다)
from IPython.display import Markdown, display
for out, _ in SETS:
    !python -m scripts.plan_eval.compare "{out}/*.summary.json" --out {out}/compare.md > /dev/null
    display(Markdown('## ' + out + '/compare.md\n\n' + open(out + '/compare.md', encoding='utf-8').read()))

## 사람이 봐야 하는 것

- 표의 숫자만으로는 **요약이 사실과 맞는지** 알 수 없다. `out/*/…review.csv` 의 `요약_사실일치` 칸에 O/X를 적는다 (기본 / 어댑터 파일을 같이 보면 바로 비교된다).
- 어댑터 쪽 `*.summary.json` 의 `adapter_info` 에 학습 데이터 SHA256·학습 판·라이브러리 판이 기록돼 있다. `revision_match` 가 false 면 어댑터를 학습한 기본 모델 판과 지금 읽은 판이 다른 것이다.
- 정확도가 비슷하면 요약이 가장 약한 고리이므로 요약 쪽 차이를 먼저 본다. 0.6B와 1.7B 중 하나를 고르는 근거는 이 표 + PC 속도 측정(`run_speed`)을 같이 본다.

In [ ]:
# 결과 내려받기: 어댑터(스모크 제외) + 측정 결과. 기본 모델은 들어 있지 않다
!rm -rf adapters/smoke
!cd /content/repo && zip -q -r plan_train_results.zip adapters out data/plan-train/train.jsonl data/plan-train/valid.jsonl
from google.colab import files
files.download('/content/repo/plan_train_results.zip')